# 03: Locality Feature Matrix

Rolls up `02`'s long `(year, position, locality)` race features into one wide row per `(year, province, city)` -- the matrix a clustering/PCA notebook runs on directly.

**Not every position is decided by the locality it's reported against.** Checked directly (Abra, 2016): `GOVERNOR`/`VICE GOVERNOR` report the same province-wide candidate slate in every city of the province (different vote splits per city, same pattern as the national races flagged in `02`); `PROVINCIAL BOARD MEMBER` and `MEMBER, HOUSE OF REPRESENTATIVES` repeat the same slate across every city sharing a district. Only `MAYOR`, `VICE MAYOR`, `COUNCILOR` are decided by the city on its own -- this is how Philippine elections are structured, not a data issue. A `GOVERNOR_margin` column describes how a city voted in a province-wide race, not a locally-decided outcome, same caveat as `02`'s for President/VP/Senator.

**Scope of this matrix:** `MAYOR`, `VICE MAYOR`, `COUNCILOR`, `GOVERNOR`, `VICE GOVERNOR`, `PROVINCIAL BOARD MEMBER`, `MEMBER, HOUSE OF REPRESENTATIVES`, `PRESIDENT`, `VICE PRESIDENT`, `SENATOR`, and party list -- positions that exist for nearly every locality nationwide. `ARMM`/`BARMM`-specific positions are excluded here (they'd be almost entirely missing outside a small subset of localities) but remain in `race_features.parquet` from `02`.

Out of scope here: joining across election years and any PSGC/population/geographic join.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("..") / "src"))

pd.set_option("display.max_columns", 40)
print("pandas:", pd.__version__)

pandas: 3.0.2


In [2]:
CONFIG = {
    "processed_dir": "../data/processed",
    "positions": [
        "MAYOR", "VICE MAYOR", "COUNCILOR",
        "GOVERNOR", "VICE GOVERNOR", "PROVINCIAL BOARD MEMBER",
        "MEMBER, HOUSE OF REPRESENTATIVES",
        "PRESIDENT", "VICE PRESIDENT", "SENATOR",
    ],
    "metrics": ["n_candidates", "total_votes", "top1_share", "top2_share", "margin", "hhi", "enc"],
}

processed_dir = Path(CONFIG["processed_dir"])
print("Config set.")

Config set.


## Load `02`'s outputs

In [3]:
race_features = pd.read_parquet(processed_dir / "race_features.parquet")
partylist_features = pd.read_parquet(processed_dir / "partylist_features.parquet")
locality_registry = pd.read_parquet(processed_dir / "locality_registry.parquet")

print("Race features:", race_features.shape)
print("Party-list features:", partylist_features.shape)
print("Locality registry:", locality_registry.shape)

Race features: (75055, 15)
Party-list features: (7587, 13)
Locality registry: (1865, 3)


## Collapse multi-district House races to one row per city

Cities split across more than one congressional district (Quezon City: 4, Davao City: 3, Cebu City, Manila's district cities, and a few others) have more than one `MEMBER, HOUSE OF REPRESENTATIVES` row per `(year, province, city)` in `race_features` -- 58 of 7,577 city-years. Pivoting directly to one row per city would keep only one district's numbers and drop the rest. Each city's districts are instead combined into a single vote-weighted figure per metric; `n_districts` records how many were combined.

In [4]:
house = race_features[race_features["position"] == "MEMBER, HOUSE OF REPRESENTATIVES"].copy()
n_multi = (house.groupby(["year", "province", "city"]).size() > 1).sum()
print(f"{n_multi} city-years have more than one congressional district.")

def collapse_house(g):
    w = g["total_votes"]
    out = {"n_districts": len(g), "n_candidates": g["n_candidates"].sum(),
           "total_votes": g["total_votes"].sum()}
    for col in ["top1_share", "top2_share", "margin", "hhi", "enc"]:
        out[col] = np.average(g[col], weights=w) if w.sum() > 0 else np.nan
    return pd.Series(out)

house_collapsed = (house.groupby(["year", "province", "city", "region"])
                    .apply(collapse_house, include_groups=False)
                    .reset_index())
house_collapsed["position"] = "MEMBER, HOUSE OF REPRESENTATIVES"

race_features_for_pivot = pd.concat([
    race_features[race_features["position"] != "MEMBER, HOUSE OF REPRESENTATIVES"],
    house_collapsed,
], ignore_index=True)
print("Rows after collapsing:", len(race_features_for_pivot),
      "(was", len(race_features), "-- difference is exactly the extra district rows folded in)")

58 city-years have more than one congressional district.


Rows after collapsing: 74941 (was 75055 -- difference is exactly the extra district rows folded in)


## Sanity check: no other position has a hidden multi-row-per-locality problem

Any other position with more than one row per `(year, province, city)` would lose rows the same way the House race did on a direct pivot. Confirmed none do.

In [5]:
scoped = race_features_for_pivot[race_features_for_pivot["position"].isin(CONFIG["positions"])]
dupe_counts = scoped.groupby(["year", "province", "city", "position"]).size()
n_dupes = (dupe_counts > 1).sum()
assert n_dupes == 0, f"{n_dupes} (year, province, city, position) combos have more than one row"
print("No duplicate (year, province, city, position) rows among the included positions.")

No duplicate (year, province, city, position) rows among the included positions.


## Build the wide matrix

One `{POSITION}_{metric}` column per position/metric combination (e.g. `MAYOR_margin`, `SENATOR_enc`), plus `PARTYLIST_*` columns from `02`. A locality/year with no recorded race for a position gets `NaN` for that position's columns -- e.g. ARMM/BARMM-region localities have no `GOVERNOR` row in some years, since the Bangsamoro Autonomous Region has its own regional executive.

In [6]:
def pivot_metrics(df, positions, metrics, index_cols):
    parts = []
    scoped = df[df["position"].isin(positions)]
    for metric in metrics:
        p = scoped.pivot_table(index=index_cols, columns="position", values=metric, aggfunc="first")
        p.columns = [f"{pos}_{metric}" for pos in p.columns]
        parts.append(p)
    return pd.concat(parts, axis=1)

race_wide = pivot_metrics(race_features_for_pivot, CONFIG["positions"], CONFIG["metrics"],
                           ["year", "province", "city"])

partylist_features = partylist_features.copy()
partylist_features["position"] = "PARTYLIST"
pl_wide = pivot_metrics(partylist_features, ["PARTYLIST"], CONFIG["metrics"],
                         ["year", "province", "city"])

locality_features = race_wide.join(pl_wide, how="outer").reset_index()
print("Locality feature matrix:", locality_features.shape)
locality_features.head()

Locality feature matrix: (9570, 80)


,year,province,city,COUNCILOR_n_candidates,GOVERNOR_n_candidates,MAYOR_n_candidates,"MEMBER, HOUSE OF REPRESENTATIVES_n_candidates",PRESIDENT_n_candidates,PROVINCIAL BOARD MEMBER_n_candidates,SENATOR_n_candidates,VICE GOVERNOR_n_candidates,VICE MAYOR_n_candidates,VICE PRESIDENT_n_candidates,COUNCILOR_total_votes,GOVERNOR_total_votes,MAYOR_total_votes,"MEMBER, HOUSE OF REPRESENTATIVES_total_votes",PRESIDENT_total_votes,PROVINCIAL BOARD MEMBER_total_votes,SENATOR_total_votes,...,VICE GOVERNOR_hhi,VICE MAYOR_hhi,VICE PRESIDENT_hhi,COUNCILOR_enc,GOVERNOR_enc,MAYOR_enc,"MEMBER, HOUSE OF REPRESENTATIVES_enc",PRESIDENT_enc,PROVINCIAL BOARD MEMBER_enc,SENATOR_enc,VICE GOVERNOR_enc,VICE MAYOR_enc,VICE PRESIDENT_enc,PARTYLIST_n_candidates,PARTYLIST_total_votes,PARTYLIST_top1_share,PARTYLIST_top2_share,PARTYLIST_margin,PARTYLIST_hhi,PARTYLIST_enc
0,2010,ABRA,BANGUED,15.0,2.0,2.0,4.0,10.0,9.0,61.0,2.0,2.0,8.0,104836.0,16902.0,19422.0,19371.0,19295.0,57487.0,144945.0,...,0.512979,0.534188,0.358602,14.027948,1.044245,1.984148,3.124203,3.888681,7.968784,21.304777,1.949399,1.872001,2.788609,187.0,14717.0,0.219066,0.106815,0.112251,0.072327,13.826069
1,2010,ABRA,BOLINEY,19.0,2.0,2.0,4.0,10.0,13.0,61.0,2.0,2.0,8.0,16100.0,1824.0,2181.0,2150.0,2120.0,6866.0,16475.0,...,0.661561,0.601324,0.293059,13.560787,1.058640,1.654304,2.517619,3.271834,7.834174,26.827933,1.511577,1.662996,3.412279,187.0,1736.0,0.314516,0.080645,0.233871,0.121056,8.260639
2,2010,ABRA,BUCAY,18.0,2.0,2.0,4.0,10.0,13.0,61.0,2.0,2.0,8.0,65450.0,7557.0,10331.0,9776.0,9525.0,24704.0,54710.0,...,0.684795,0.500166,0.354190,16.900012,1.061597,1.972448,2.835062,3.964144,7.319257,19.612672,1.460290,1.999337,2.823340,187.0,6408.0,0.148408,0.119226,0.029182,0.056164,17.805109
3,2010,ABRA,BUCLOC,31.0,2.0,2.0,4.0,10.0,13.0,61.0,2.0,2.0,8.0,10403.0,1084.0,1364.0,1358.0,1324.0,4092.0,9913.0,...,0.594017,0.552560,0.517155,24.741227,1.064694,1.600000,2.175692,3.799584,7.255439,25.913630,1.683454,1.809759,1.933655,187.0,1112.0,0.201439,0.161871,0.039568,0.094944,10.532563
4,2010,ABRA,DAGUIOMAN,17.0,2.0,3.0,4.0,10.0,13.0,61.0,2.0,2.0,8.0,8313.0,892.0,1133.0,1083.0,992.0,2902.0,5504.0,...,0.509731,0.500011,0.304017,16.033722,1.076720,2.013690,3.282507,3.493826,7.786621,27.082517,1.961818,1.999958,3.289291,187.0,705.0,0.136170,0.120567,0.015603,0.059601,16.778348


## Fill in region and save

`region` is constant per province, attached from `01`'s locality registry -- the pivot itself only keeps numeric metrics.

In [7]:
region_lookup = locality_registry.set_index(["province", "city"])["region"]
locality_features["region"] = locality_features.set_index(["province", "city"]).index.map(region_lookup)

cols = ["year", "province", "city", "region"] + [c for c in locality_features.columns
                                                    if c not in ("year", "province", "city", "region")]
locality_features = locality_features[cols]

n_missing_region = locality_features["region"].isna().sum()
print(f"{n_missing_region} of {len(locality_features)} rows still missing region after lookup.")
locality_features.to_parquet(processed_dir / "locality_features.parquet", index=False)
print("Saved locality_features.parquet:", locality_features.shape)

0 of 9570 rows still missing region after lookup.
Saved locality_features.parquet: (9570, 81)


## Sanity checks

Range checks on every share/margin/enc column, plus a coverage check: near-universal positions exist almost everywhere, and region-only positions don't leak outside their region.

In [8]:
share_cols = [c for c in locality_features.columns if c.endswith(("top1_share", "top2_share", "margin"))]
for col in share_cols:
    vals = locality_features[col].dropna()
    assert vals.between(0, 1 + 1e-9).all(), f"{col} has a value outside [0, 1]"

enc_cols = [c for c in locality_features.columns if c.endswith("_enc")]
for col in enc_cols:
    vals = locality_features[col].dropna()
    assert (vals >= 1 - 1e-9).all(), f"{col} has a value below 1"

print(f"All {len(share_cols)} share/margin columns and {len(enc_cols)} enc columns passed range checks.")

coverage = locality_features[[f"{p}_top1_share" for p in CONFIG["positions"]] + ["PARTYLIST_top1_share"]].notna().mean()
print("\nFraction of locality-years with a recorded race, by position:")
display(coverage.sort_values().to_frame("coverage"))

All 33 share/margin columns and 11 enc columns passed range checks.

Fraction of locality-years with a recorded race, by position:


,coverage
PRESIDENT_top1_share,0.500522
VICE PRESIDENT_top1_share,0.500522
VICE GOVERNOR_top1_share,0.773772
GOVERNOR_top1_share,0.773772
PROVINCIAL BOARD MEMBER_top1_share,0.773772
"MEMBER, HOUSE OF REPRESENTATIVES_top1_share",0.790178
PARTYLIST_top1_share,0.792790
SENATOR_top1_share,0.841484
COUNCILOR_top1_share,0.936991
VICE MAYOR_top1_share,0.950888


## A quick look before handing this to clustering

A quick check that the matrix looks like real Philippine election data before it becomes clustering input: Mayor races should be more competitive on average than the uncontested-heavy small-council races, and national-race columns should show regional structure (same nationwide ballot, so the spread is pure locality-level variation in support).

`PRESIDENT_top1_share` is only ~50% covered -- not missing data, but because the Philippines elects a President once every two NLE cycles (2010, 2016, 2022 have a presidential race; 2013, 2019, 2025 are midterms without one). The region breakdown below picks the latest year with an actual presidential race rather than the latest year overall, which would otherwise be an all-`NaN` midterm.

In [9]:
summary_cols = ["MAYOR_margin", "MAYOR_enc", "COUNCILOR_margin", "COUNCILOR_enc",
                 "PRESIDENT_top1_share", "PARTYLIST_enc"]
display(locality_features[summary_cols].describe().T)

years_with_president = locality_features.loc[locality_features["PRESIDENT_top1_share"].notna(), "year"]
print("Years with a presidential race:", sorted(years_with_president.unique()))
latest_presidential_year = years_with_president.max()

print(f"\nMedian PRESIDENT_top1_share by region, {latest_presidential_year}:")
by_region = (locality_features[locality_features["year"] == latest_presidential_year]
             .groupby("region")["PRESIDENT_top1_share"].median().sort_values(ascending=False))
display(by_region.to_frame(f"median top1_share ({latest_presidential_year})"))

,count,mean,std,min,25%,50%,75%,max
MAYOR_margin,9110.0,0.361852,0.336268,0.000000,0.093688,0.228605,0.557884,1.000000
MAYOR_enc,9110.0,1.850324,0.556196,1.000000,1.546849,1.937967,1.999907,6.125626
COUNCILOR_margin,8967.0,0.007612,0.009220,0.000000,0.002166,0.005199,0.010396,0.394810
COUNCILOR_enc,8967.0,16.405838,4.802436,3.769008,13.724732,15.956160,18.393413,84.237191
PRESIDENT_top1_share,4790.0,0.548784,0.172154,0.231545,0.417591,0.507790,0.652400,1.000000
PARTYLIST_enc,7587.0,11.151600,8.163317,1.004651,4.600885,8.632463,16.104565,51.132807


Years with a presidential race: [np.int64(2010), np.int64(2016), np.int64(2022)]

Median PRESIDENT_top1_share by region, 2022:


,median top1_share (2022)
region,
REGION I,0.928718
CORDILLERA ADMINISTRATIVE REGION,0.897986
REGION II,0.870132
REGION XI,0.797711
REGION V,0.789049
REGION X,0.708941
BARMM,0.697115
CARAGA,0.692529
REGION XII,0.669718


## Summary

`locality_features.parquet` holds one row per `(year, province, city)` with `{position}_{metric}` columns for the ten broadly-applicable positions plus party list -- ready for clustering/PCA. Multi-district House races were combined into a single vote-weighted figure per city instead of truncated; ARMM/BARMM-specific positions were scoped out rather than left as mostly-empty columns.

**Next:** clustering/dimensionality reduction on this matrix -- column selection, `NaN` handling, and whether to model each election year separately or pool them are decisions for `04`.